In [0]:
%sql
CREATE OR REPLACE TABLE krishi_dhwani.gold.market_price_trends AS
SELECT
    crop,
    AVG(price_per_kg_inr) AS avg_price_7d,
    MAX(price_per_kg_inr) AS max_price_7d,
    MIN(price_per_kg_inr) AS min_price_7d,
    CASE 
        WHEN AVG(price_per_kg_inr) > 40 THEN 'Profitable'
        ELSE 'Below Average'
    END AS trend
FROM krishi_dhwani.silver.market_prices
WHERE date >= date_sub(current_date(), 7)
GROUP BY crop

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE krishi_dhwani.gold.farmer_soil_health AS
SELECT
    farmer_id,
    name,
    pincode,
    state,
    language,
    land_acres,
    soil_type,
    ph,
    nitrogen_kg_ha,
    phosphorus_kg_ha,
    potassium_kg_ha,
    primary_crop,
    CASE
        WHEN ph BETWEEN 6.0 AND 7.5 THEN 'Good'
        WHEN ph BETWEEN 5.5 AND 8.0 THEN 'Acceptable'
        ELSE 'Needs Amendment'
    END AS soil_health_status
FROM krishi_dhwani.bronze.farmer_profiles_raw

num_affected_rows,num_inserted_rows


In [0]:
spark.table("krishi_dhwani.gold.farmer_soil_health").show()

+---------+--------------+-------+---------+--------+----------+----------+---+--------------+----------------+---------------+------------+------------------+
|farmer_id|          name|pincode|    state|language|land_acres| soil_type| ph|nitrogen_kg_ha|phosphorus_kg_ha|potassium_kg_ha|primary_crop|soil_health_status|
+---------+--------------+-------+---------+--------+----------+----------+---+--------------+----------------+---------------+------------+------------------+
|     F001|Rajinder Singh| 141001|   Punjab| punjabi|       5.0|     Loamy|6.8|           180|              45|            220|       Wheat|              Good|
|     F002|   Sunita Devi| 302001|Rajasthan|   hindi|       2.5|Sandy Loam|7.2|           120|              30|            150|     Mustard|              Good|
+---------+--------------+-------+---------+--------+----------+----------+---+--------------+----------------+---------------+------------+------------------+



In [0]:
%pip install sentence-transformers faiss-cpu PyMuPDF requests gradio pydantic

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
dbutils.library.restartPython()

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

In [0]:
index_path = "/Volumes/krishi_dhwani/silver/faiss_index/icar.index"
metadata_path = "/Volumes/krishi_dhwani/silver/faiss_index/metadata.pkl"

_faiss_index = faiss.read_index(index_path)

with open(metadata_path, "rb") as f:
    _faiss_data = pickle.load(f)

In [0]:
_embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [0]:
def search_icar_knowledge(query: str, top_k: int = 3) -> str:
    """Search ICAR agricultural knowledge base."""
    
    query_vec = _embed_model.encode([query]).astype("float32")
    faiss.normalize_L2(query_vec)

    distances, indices = _faiss_index.search(query_vec, top_k)

    results = []
    for idx, score in zip(indices[0], distances[0]):
        if idx >= 0:
            chunk = _faiss_data["chunks"][idx]
            source = _faiss_data["metadata"][idx]["source"]
            results.append(f"[Source: {source}]\n{chunk}")

    return "\n\n---\n\n".join(results)

In [0]:
print(search_icar_knowledge("How to improve wheat yield?"))

[Source: dbfs:/Volumes/krishi_dhwani/bronze/icar_volume/farmerbook.pdf]
s
33
24
Cotton
36
50
Cowpea
19
3
Fenugreek
29
25
Garlic
28
6
Gram
69
57
Groundnut
20
40
Jowar
55
34
Lucerne
16
27
Maize
41
36
Onion
33
23
Potato
46
4
Sunflower
33
20
Wheat
35
24
Lay out of Sprinkler Irrigation System
Drip Irrigation
Water is applied
At low rate
Over a long period of time.
At frequent intervals
Directly into the plant’s root zone
16
General Conditions for Cultivation of Crops
Farmer’s Handbook on Basic Agriculture
Drip irrigation
Water is conveyed under pressure through a pip

---

[Source: dbfs:/Volumes/krishi_dhwani/bronze/icar_volume/farmerbook.pdf]
ize, Sorghum, Wheat, Jowar
Flowers
Carnation, Jasmine, Marigold
Oilseeds
Groundnut, Mustard, Sunflower
Vegetables Onion, Potato, Radish, Carrot
Fodders
Asparagus, Pastures
Pulses
Gram, Pigeon pea, Beans
Plantation Coffee, Rubber, Tamarind
Fibre
Cotton, Sesame
Spices
Cardamom
Response of different crops to sprinkler irrigation
Crop 
Water  
saving 

In [0]:
def get_farmer_soil_health(farmer_id: str) -> dict:
    rows = spark.sql(f"""
        SELECT * FROM krishi_dhwani.gold.farmer_soil_health
        WHERE farmer_id = '{farmer_id}'
    """).collect()

    if rows:
        return rows[0].asDict()

    return {"error": f"Farmer {farmer_id} not found"}

In [0]:
print(get_farmer_soil_health("F001"))

{'farmer_id': 'F001', 'name': 'Rajinder Singh', 'pincode': '141001', 'state': 'Punjab', 'language': 'punjabi', 'land_acres': 5.0, 'soil_type': 'Loamy', 'ph': 6.8, 'nitrogen_kg_ha': 180, 'phosphorus_kg_ha': 45, 'potassium_kg_ha': 220, 'primary_crop': 'Wheat', 'soil_health_status': 'Good'}


In [0]:
def get_weather_forecast(pincode: str) -> list:
    rows = spark.sql(f"""
        SELECT forecast_date, rainfall_mm, temp_max_c, condition
        FROM krishi_dhwani.silver.weather
        WHERE pincode = '{pincode}'
        ORDER BY forecast_date
        LIMIT 7
    """).collect()

    return [r.asDict() for r in rows]

In [0]:
print(get_weather_forecast("141001"))

[{'forecast_date': '2026-04-07', 'rainfall_mm': 33.0, 'temp_max_c': 33.4, 'condition': 'Clear'}, {'forecast_date': '2026-04-08', 'rainfall_mm': 21.7, 'temp_max_c': 40.9, 'condition': 'Partly Cloudy'}, {'forecast_date': '2026-04-09', 'rainfall_mm': 33.7, 'temp_max_c': 28.9, 'condition': 'Thunderstorm'}, {'forecast_date': '2026-04-10', 'rainfall_mm': 28.6, 'temp_max_c': 27.9, 'condition': 'Partly Cloudy'}, {'forecast_date': '2026-04-11', 'rainfall_mm': 13.0, 'temp_max_c': 41.1, 'condition': 'Rain'}, {'forecast_date': '2026-04-12', 'rainfall_mm': 9.4, 'temp_max_c': 41.2, 'condition': 'Rain'}, {'forecast_date': '2026-04-13', 'rainfall_mm': 17.8, 'temp_max_c': 25.9, 'condition': 'Rain'}]


In [0]:
def get_market_price(crop: str) -> dict:
    rows = spark.sql(f"""
        SELECT * FROM krishi_dhwani.gold.market_price_trends
        WHERE lower(crop) = lower('{crop}')
    """).collect()

    if rows:
        return rows[0].asDict()

    return {"error": f"No price data for {crop}"}

In [0]:
print(get_market_price("Wheat"))

{'crop': 'Wheat', 'avg_price_7d': 34.96875, 'max_price_7d': 45.25, 'min_price_7d': 20.14, 'trend': 'Below Average'}


In [0]:
import requests
import json

In [0]:
DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl")
DATABRICKS_TOKEN = "<YOUR_DATABRICKS_API_TOKEN>"

In [0]:
def run_krishi_agent(english_query: str, farmer_id: str) -> str:

    # 1. Fetch farmer context
    farmer = get_farmer_soil_health(farmer_id)
    pincode = farmer.get("pincode", "110001")
    primary_crop = farmer.get("primary_crop", "Wheat")

    # 2. Tool calls (YOU control them here)
    icar_context = search_icar_knowledge(english_query)
    weather = get_weather_forecast(pincode)
    market = get_market_price(primary_crop)

    # 3. Prompt
    system_prompt = """You are Krishi Dhwani, an expert agricultural advisor for Indian farmers.
You have access to ICAR research, weather forecasts, and market prices.
Give specific, actionable advice.
Always cite sources.
Keep answer concise (3-5 sentences).
End with a clear recommendation."""

    context_block = f"""
FARMER PROFILE:
{json.dumps(farmer, indent=2)}

7-DAY WEATHER (Pincode {pincode}):
{json.dumps(weather, indent=2)}

MARKET PRICE TREND ({primary_crop}):
{json.dumps(market, indent=2)}

ICAR KNOWLEDGE:
{icar_context}
"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Context:\n{context_block}\n\nQuestion: {english_query}"}
    ]

    response = call_llama_maverick(messages)
    return response["choices"][0]["message"]["content"]

In [0]:
%pip install databricks-langchain

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
dbutils.library.restartPython()

com.databricks.backend.common.rpc.CommandSkippedException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:141)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:136)
	at scala.collection.immutable.Range.foreach(Range.scala:192)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:724)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:442)
	at scala.Option.getOrElse(Option.scala:201)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:442)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.cancelExecution(ExecutionContextManagerV1.scala:495)
	at com.databricks.spark.chauffeur.ChauffeurState.$anonfun$process$1(ChauffeurState.scala:725)
	at com.data

In [0]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(
    endpoint="databricks-llama-4-maverick",  # you confirmed this exists
    temperature=0.3,
    max_tokens=200
)

response = llm.invoke("Should I sow mustard this week in North India?")
print(response.content)

To determine whether you should sow mustard this week in North India, we need to consider a couple of key factors: the current weather conditions and the typical sowing season for mustard in North India.

1. **Sowing Season for Mustard in North India**: Mustard is typically grown as a rabi crop in India. The rabi season is from October to March, with the sowing usually happening in October-November. Mustard requires a cool season to grow, and it thrives in the winter months.

2. **Weather Conditions**: North India's climate varies significantly from the onset of winter to the end. By the time it's mid-to-late winter, the temperatures are quite low, especially in the mornings and evenings. Mustard requires a certain period of cool weather to mature but also needs a relatively warmer period for germination and initial growth.

3. **Current Timing**: Without knowing the exact week you're referring to, a general guideline is that mustard can be sown from late September


In [0]:
def call_llama_maverick(messages: list) -> dict:
    
    # Convert messages → single prompt
    prompt = ""
    for m in messages:
        role = m["role"]
        content = m["content"]
        prompt += f"{role.upper()}:\n{content}\n\n"

    response = llm.invoke(prompt)

    return {
        "choices": [
            {
                "message": {
                    "content": response.content
                }
            }
        ]
    }

In [0]:
print(run_krishi_agent(
    "Should I sow mustard this week?",
    "F002"
))

Sunita Devi ji, based on the 7-day weather forecast for your area (Pincode 302001), there is significant rainfall predicted for the next few days, which may not be ideal for sowing mustard. According to the ICAR knowledge, mustard is typically sown in October-November or September-October in Rajasthan (Source: Farmer's Handbook on Basic Agriculture). Considering the current weather and time of the year (April), it's not the suitable time for sowing mustard. 
Recommendation: Delay sowing mustard and consider alternative crops suitable for the current weather conditions, such as summer crops like cowpea or vegetables like onion or radish.


In [0]:
import requests
import base64

In [0]:
SARVAM_API_KEY = "<YOUR_SARVAM_API_KEY>"
SARVAM_BASE = "https://api.sarvam.ai"

In [0]:
def sarvam_stt(audio_bytes: bytes, language_code: str = "hi-IN") -> str:

    files = {
        "file": ("audio.wav", audio_bytes, "audio/wav")  # ✅ ADD THIS
    }

    data = {
        "model": "saaras:v3",
        "language_code": language_code
    }

    headers = {
        "api-subscription-key": SARVAM_API_KEY
    }

    resp = requests.post(
        f"{SARVAM_BASE}/speech-to-text",
        files=files,
        data=data,
        headers=headers
    )

    print(resp.text)  # keep for debug
    resp.raise_for_status()

    return resp.json().get("transcript", "")

In [0]:
def sarvam_translate_to_english(text: str, source_language: str = "hi-IN") -> str:

    headers = {
        "api-subscription-key": SARVAM_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "input": text,
        "source_language_code": source_language,
        "target_language_code": "en-IN",
        "model": "mayura:v1",
        "enable_preprocessing": True
    }

    resp = requests.post(f"{SARVAM_BASE}/translate", json=payload, headers=headers)
    resp.raise_for_status()

    return resp.json().get("translated_text", text)

In [0]:
def sarvam_translate_from_english(text: str, target_language: str = "hi-IN") -> str:

    headers = {
        "api-subscription-key": SARVAM_API_KEY,
        "Content-Type": "application/json"
    }

    payload = {
        "input": text,
        "source_language_code": "en-IN",
        "target_language_code": target_language,
        "model": "mayura:v1",
        "enable_preprocessing": True
    }

    resp = requests.post(f"{SARVAM_BASE}/translate", json=payload, headers=headers)
    resp.raise_for_status()

    return resp.json().get("translated_text", text)

In [0]:
def sarvam_tts(text: str, language_code: str = "hi-IN") -> bytes:
    headers = {
        "api-subscription-key": SARVAM_API_KEY,
        "Content-Type": "application/json"
    }

    text = text.replace("\n", " ")[:500]

    payload = {
        "inputs": [text],
        "target_language_code": language_code,
        "speaker": "priya",      # valid lowercase voice
        "model": "bulbul:v3"     # current TTS model
    }

    resp = requests.post(f"{SARVAM_BASE}/text-to-speech", json=payload, headers=headers)
    print(resp.text)
    resp.raise_for_status()

    audio_b64 = resp.json()["audios"][0]
    return base64.b64decode(audio_b64)

In [0]:
LANGUAGE_MAP = {
    "Hindi": "hi-IN",
    "Punjabi": "pa-IN",
    "Tamil": "ta-IN",
    "Telugu": "te-IN",
    "Marathi": "mr-IN",
    "Kannada": "kn-IN",
    "Gujarati": "gu-IN",
    "Bengali": "bn-IN",
    "Odia": "od-IN"
}
def full_voice_pipeline(audio_bytes: bytes, farmer_id: str, language: str = "Hindi"):

    lang_code = LANGUAGE_MAP.get(language, "hi-IN")

    # 1. STT
    regional_text = sarvam_stt(audio_bytes, lang_code)
    print("Farmer said:", regional_text)

    # 2. Translate → English
    english_query = sarvam_translate_to_english(regional_text, lang_code)
    print("English:", english_query)

    # 3. Agent
    english_answer = run_krishi_agent(english_query, farmer_id)
    print("Answer:", english_answer)

    # 4. Translate back
    regional_answer = sarvam_translate_from_english(english_answer, lang_code)
    print("Regional:", regional_answer)

    # 5. TTS
    audio_out = sarvam_tts(regional_answer, lang_code)

    # 🔥 6. SAVE FILE (IMPORTANT FIX)
    output_path = "/Workspace/Users/cs5210611@iitd.ac.in/test.wav"
    
    with open(output_path, "wb") as f:
        f.write(audio_out)

    print(f"Audio saved at: {output_path}")

    return audio_out, regional_answer, english_answer

In [0]:
text = "क्या मुझे इस हफ्ते सरसों बोनी चाहिए?"

eng = sarvam_translate_to_english(text)
print("English:", eng)

ans = run_krishi_agent(eng, "F002")

final = sarvam_translate_from_english(ans)
print("Final:", final)

English: Should I sow mustard this week?
Final: सुनीता देवी जी, अगले 7 दिनों के मौसम पूर्वानुमान को ध्यान में रखते हुए, कई दिनों तक वर्षा की उच्च संभावना है, जो सरसों के बीज बोने के लिए आदर्श नहीं हो सकती है क्योंकि यह अंकुरण के लिए शुष्क परिस्थितियों को पसंद करती है (स्रोत: किसान पुस्तिका पर बुनियादी कृषि, आई.सी.ए.आर.)। इसके अलावा, सरसों के लिए बाजार मूल्य का रुझान वर्तमान में औसत से कम है। मिट्टी की स्वास्थ्य स्थिति अच्छी होने और मिट्टी सरसों के लिए उपयुक्त होने के कारण, आप अत्यधिक नमी से बचने के लिए बुवाई में एक सप्ताह की देरी पर विचार कर सकते हैं। अनुशंसा: अत्यधिक वर्षा से बचने के लिए सरसों की बुवाई एक सप्ताह विलंबित करें।


In [0]:
with open("/Workspace/Users/cs5210611@iitd.ac.in/Ber Sarai 12.wav", "rb") as f:
    audio_bytes = f.read()

audio_out, regional, english = full_voice_pipeline(audio_bytes, "F002", "Hindi")

{"request_id":"20260407_d3bca3cb-6a02-45c4-8fee-fe51bc8baffb","transcript":"क्या मुझे इस हफ़्ते सरसों बोनी चाहिए?","language_code":"hi-IN"}
Farmer said: क्या मुझे इस हफ़्ते सरसों बोनी चाहिए?
English: Should I sow mustard seeds this week?
Answer: Sunita Devi ji, based on the 7-day weather forecast for your area (Pincode 302001), it is expected to receive significant rainfall from April 8th to 11th, which is beneficial for sowing mustard seeds. However, considering the current market price trend for mustard is "Below Average" (avg_price_7d: ₹31.83), it might not be the best time to sow. According to ICAR's Farmer's Handbook on Basic Agriculture, the critical stages for irrigation in oilseeds like mustard are flowering and pod development. 
Recommendation: Delay sowing mustard seeds until the market price trend improves or consider alternative crops like Gram or Cotton, which have shown positive response to irrigation (Source: ICAR Farmer's Handbook).
Regional: सुनीता देवी जी, आपके क्षेत्

{"request_id":"20260407_a562f914-4f0b-4d5c-b6a8-478eafc276f9","audios":["..."]}
Audio saved at: /Workspace/Users/cs5210611@iitd.ac.in/test.wav
